# Conformal TB Triage: Download Datasets to Google Drive

**Runtime:** CPU (no GPU needed — save your GPU quota for embedding extraction).

**Safe to re-run.** Every cell checks whether its work is already done and skips if so. If a session disconnects, just re-run from the top.

**Downloads:** Shenzhen, Montgomery, TBX11K, CheXpert (5K subset), Mendeley Pakistan.

In [ ]:
# ── Cell 1: Setup (run this first every time) ──────────────────────────
# Mounts Drive, installs kaggle, defines all shared helpers.
# Safe to re-run: mount is idempotent, pip is quiet, helpers are just defs.

import os, sys, shutil, hashlib, random, subprocess
from pathlib import Path
from datetime import datetime, timezone

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install kaggle CLI
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

# ── Paths ──
DRIVE_ROOT = Path('/content/drive/MyDrive/conformal-tb-triage')
RAW = DRIVE_ROOT / 'data' / 'raw'
EMBEDDINGS = DRIVE_ROOT / 'data' / 'interim' / 'embeddings'
TMP = Path('/content/tmp_downloads')  # fast local disk for staging

for d in [RAW/'tbx11k', RAW/'shenzhen_montgomery', RAW/'mendeley_pakistan',
          RAW/'jsrt', RAW/'padchest_subset', RAW/'chexpert_subset',
          RAW/'nih_cxr14', RAW/'tb_portals', EMBEDDINGS, TMP]:
    d.mkdir(parents=True, exist_ok=True)

# ── Helpers ──
def count_images(d, exts=('.png','.jpg','.jpeg','.dcm')):
    d = Path(d)
    if not d.exists(): return 0
    return sum(len(list(d.rglob(f'*{e}'))) for e in exts)

def run_live(cmd):
    """Run a shell command with real-time line-by-line output in Colab."""
    print(f'  $ {cmd}', flush=True)
    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(f'  {line}', end='', flush=True)
    process.wait()
    return process.returncode == 0

def kaggle_download(slug, dest_local, unzip=True):
    """Download a Kaggle dataset with live output."""
    dest_local = Path(dest_local)
    dest_local.mkdir(parents=True, exist_ok=True)
    cmd = f'kaggle datasets download -d {slug} -p {dest_local}'
    if unzip:
        cmd += ' --unzip'
    return run_live(cmd)

def copy_to_drive(src, dst):
    """Copy files from local Colab disk to Google Drive with progress."""
    src, dst = Path(src), Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    files = [f for f in src.rglob('*') if f.is_file()]
    total = len(files)
    print(f'  Copying {total} files to Drive...', flush=True)
    for i, f in enumerate(files):
        rel = f.relative_to(src)
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(f, target)
        if (i + 1) % 200 == 0 or (i + 1) == total:
            print(f'  ... {i+1}/{total} files', flush=True)

print('Setup complete.')
print(f'Drive root: {DRIVE_ROOT}')
print(f'Staging:    {TMP}')

In [ ]:
# ── Cell 2: Kaggle API token ──────────────────────────────────────────
# Tries multiple auth methods since Kaggle changed their token format.
# Uses getpass so the token never appears in notebook output.

import json
from getpass import getpass

# Upgrade to latest kaggle package (older versions don't know KGAT_ tokens)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'],
               capture_output=True)

token = getpass('Paste your Kaggle API token (KGAT_...): ')

# Method 1: KAGGLE_KEY env var (works on newer kaggle package)
os.environ['KAGGLE_KEY'] = token
os.environ.pop('KAGGLE_USERNAME', None)  # clear old-style auth if present

# Method 2: Write to kaggle.json in new token format
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
# Try new format first, fall back to old format
for fmt in [{"token": token}, {"key": token}, {"username": "_", "key": token}]:
    with open(kaggle_json, 'w') as f:
        json.dump(fmt, f)
    os.chmod(kaggle_json, 0o600)

    r = subprocess.run(['kaggle', 'datasets', 'list', '-s', 'chest-xray', '--max-size', '1'],
                       capture_output=True, text=True)
    if r.returncode == 0 and 'error' not in r.stderr.lower():
        print(f'Kaggle authenticated (format: {list(fmt.keys())})')
        break
else:
    # Method 3: Set both env vars in case it wants old-style
    os.environ['KAGGLE_USERNAME'] = 'token'
    os.environ['KAGGLE_KEY'] = token
    r = subprocess.run(['kaggle', 'datasets', 'list', '-s', 'chest-xray', '--max-size', '1'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print('Kaggle authenticated (env vars)')
    else:
        print(f'All auth methods failed.')
        print(f'stderr: {r.stderr.strip()[:300]}')
        print(f'stdout: {r.stdout.strip()[:300]}')
        print()
        print('Troubleshooting:')
        print('  1. Go to kaggle.com > Settings > API')
        print('  2. If you see "Create New Token" — click it, it may download a kaggle.json file')
        print('  3. If you got a kaggle.json, upload it with the cell below instead')

In [ ]:
# ── Cell 2b (fallback): Upload kaggle.json directly ──────────────────
# Only run this if Cell 2 failed.
# Go to kaggle.com > Settings > API > "Create New Token" — this downloads
# a kaggle.json file. Then run this cell and upload that file.

# from google.colab import files
# print('Upload your kaggle.json file:')
# uploaded = files.upload()
# kaggle_dir = Path.home() / '.kaggle'
# kaggle_dir.mkdir(exist_ok=True)
# with open(kaggle_dir / 'kaggle.json', 'wb') as f:
#     f.write(uploaded['kaggle.json'])
# os.chmod(kaggle_dir / 'kaggle.json', 0o600)
# r = subprocess.run(['kaggle', 'datasets', 'list', '-s', 'chest-xray', '--max-size', '1'],
#                    capture_output=True, text=True)
# print('Authenticated.' if r.returncode == 0 else f'Still failing: {r.stderr.strip()[:200]}')

In [ ]:
# ── Cell 3: Download Shenzhen + Montgomery ────────────────────────────
# Source: Kaggle mirror (NLM direct links are unreliable as of 2026).

dest = RAW / 'shenzhen_montgomery'
n = count_images(dest)

if n >= 700:
    print(f'Already downloaded ({n} images). Skipping.')
else:
    print('Downloading Shenzhen + Montgomery...', flush=True)
    staging = TMP / 'shen_mont'
    shutil.rmtree(staging, ignore_errors=True)

    slugs = [
        'raddar/tuberculosis-chest-xrays-shenzhen',
        'kmader/pulmonary-chest-xray-abnormalities',
        'tolgadincer/labeled-chest-xray-images',
    ]

    downloaded = False
    for slug in slugs:
        print(f'\n── Trying Kaggle: {slug}', flush=True)
        if kaggle_download(slug, staging):
            n_staged = count_images(staging)
            if n_staged > 100:
                print(f'\n  Got {n_staged} images. Copying to Drive...', flush=True)
                copy_to_drive(staging, dest)
                downloaded = True
                break
            else:
                print(f'  Only {n_staged} images — trying next.', flush=True)
                shutil.rmtree(staging, ignore_errors=True)
        else:
            shutil.rmtree(staging, ignore_errors=True)

    if not downloaded:
        print('\n── Kaggle mirrors failed. Trying NLM direct...', flush=True)
        staging.mkdir(parents=True, exist_ok=True)
        for name, url in [
            ('ChinaSet_AllFiles.zip',
             'https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/ChinaSet_AllFiles.zip'),
            ('MontgomerySet.zip',
             'https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet.zip'),
        ]:
            zip_path = staging / name
            print(f'\n  Downloading {name}:', flush=True)
            run_live(f'curl -L --progress-bar -o {zip_path} "{url}"')
            if zip_path.exists() and zip_path.stat().st_size > 1_000_000:
                with open(zip_path, 'rb') as f:
                    magic = f.read(2)
                if magic == b'PK':
                    print(f'  Valid zip ({zip_path.stat().st_size/1e6:.1f} MB). Extracting:', flush=True)
                    run_live(f'unzip -o {zip_path} -d {staging}')
                else:
                    print(f'  Not a zip file (HTML redirect). Skipping.', flush=True)
            else:
                size = zip_path.stat().st_size if zip_path.exists() else 0
                print(f'  Too small ({size} bytes). Skipping.', flush=True)

        n_staged = count_images(staging)
        if n_staged > 100:
            print(f'\n  Got {n_staged} images. Copying to Drive...', flush=True)
            copy_to_drive(staging, dest)
            downloaded = True

    shutil.rmtree(staging, ignore_errors=True)
    n_final = count_images(dest)
    if n_final >= 700:
        print(f'\nDone: {n_final} images on Drive.')
    else:
        print(f'\nOnly {n_final} images. Manual download needed.')
        print('Search Kaggle for "shenzhen montgomery chest xray"')
        print(f'Place files in: {dest}')

In [ ]:
# ── Cell 3b: Unpack manually uploaded Shenzhen/Montgomery ─────────────
# You uploaded archive.zip to Drive at:
#   conformal-tb-triage/data/raw/shenzhen_montgomery/images/archive.zip
# This cell extracts it and moves images to the right place.

dest = RAW / 'shenzhen_montgomery'
archive = dest / 'images' / 'archive.zip'

if not archive.exists():
    print(f'No archive.zip found at {archive}')
    print(f'Current contents of {dest}:')
    for p in sorted(dest.rglob('*'))[:20]:
        print(f'  {p.relative_to(dest)}')
else:
    size_mb = archive.stat().st_size / 1e6
    print(f'Found archive.zip ({size_mb:.1f} MB). Extracting to local disk...', flush=True)

    # Extract to fast local disk first
    staging = TMP / 'shen_mont_manual'
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True, exist_ok=True)

    run_live(f'unzip -o "{archive}" -d {staging}')

    # See what we got
    n_staged = count_images(staging)
    print(f'\nExtracted {n_staged} images.', flush=True)

    if n_staged > 0:
        # List top-level structure so we know what's inside
        top_items = sorted(set(p.parts[len(staging.parts)] for p in staging.rglob('*') if p.is_file()))
        print(f'Top-level contents: {top_items[:10]}', flush=True)

        # Copy images to the shenzhen_montgomery folder on Drive
        print('Copying to Drive...', flush=True)
        copy_to_drive(staging, dest)
        shutil.rmtree(staging, ignore_errors=True)

        n_final = count_images(dest)
        print(f'\nDone: {n_final} images in shenzhen_montgomery/ on Drive.')

        # Optionally remove the zip to save Drive space
        # archive.unlink()
        # print(f'Deleted archive.zip to save {size_mb:.0f} MB on Drive.')
    else:
        print('No images found in archive. Listing all extracted files:')
        for p in sorted(staging.rglob('*'))[:30]:
            print(f'  {p.relative_to(staging)}')
        shutil.rmtree(staging, ignore_errors=True)

In [ ]:
# ── Cell 3d: Unpack manually uploaded Montgomery ──────────────────────
# You uploaded NLM-MontgomeryCXRSet.zip to Drive at:
#   conformal-tb-triage/data/raw/shenzhen_montgomery/NLM-MontgomeryCXRSet.zip

dest = RAW / 'shenzhen_montgomery'
archive = dest / 'NLM-MontgomeryCXRSet.zip'

if not archive.exists():
    print(f'No NLM-MontgomeryCXRSet.zip found at {archive}')
    print('Check the filename and location on Drive.')
else:
    size_mb = archive.stat().st_size / 1e6
    print(f'Found NLM-MontgomeryCXRSet.zip ({size_mb:.1f} MB). Extracting...', flush=True)

    staging = TMP / 'montgomery_manual'
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True, exist_ok=True)

    run_live(f'unzip -o "{archive}" -d {staging}')

    n_staged = count_images(staging)
    print(f'\nExtracted {n_staged} images.', flush=True)

    if n_staged > 0:
        print('Copying to Drive...', flush=True)
        copy_to_drive(staging, dest)
        shutil.rmtree(staging, ignore_errors=True)

        n_total = count_images(dest)
        print(f'\nDone. Total images in shenzhen_montgomery/: {n_total}')
        print(f'  (Expected ~800: 662 Shenzhen + 138 Montgomery)')
    else:
        print('No images found in zip. Contents:')
        for p in sorted(staging.rglob('*'))[:30]:
            print(f'  {p.relative_to(staging)}')
        shutil.rmtree(staging, ignore_errors=True)

In [ ]:
# ── Cell 3c: Download Montgomery only ─────────────────────────────────
# Shenzhen is already on Drive (662 images). Montgomery is 138 images,
# separate dataset. Tries multiple sources.

dest = RAW / 'shenzhen_montgomery'
# Check if Montgomery is already there (look for its distinctive filenames)
montgomery_count = len(list(dest.rglob('*MC*'))) + len(list(dest.rglob('*montgomery*')))
total = count_images(dest)

if total >= 790:
    print(f'Already have {total} images (Shenzhen + Montgomery). Skipping.')
elif montgomery_count > 50:
    print(f'Montgomery already present ({montgomery_count} files). Total: {total}.')
else:
    print(f'Currently have {total} images (Shenzhen only). Downloading Montgomery...', flush=True)
    staging = TMP / 'montgomery'
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True, exist_ok=True)

    got_it = False

    # Attempt 1: NLM direct URL
    print('\n── Attempt 1: NLM direct download', flush=True)
    zip_path = staging / 'MontgomerySet.zip'
    run_live(f'curl -L --progress-bar -o {zip_path} '
             f'"https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet.zip"')

    if zip_path.exists() and zip_path.stat().st_size > 500_000:
        with open(zip_path, 'rb') as f:
            magic = f.read(2)
        if magic == b'PK':
            print(f'  Valid zip ({zip_path.stat().st_size/1e6:.1f} MB). Extracting:', flush=True)
            run_live(f'unzip -o {zip_path} -d {staging}')
            n = count_images(staging)
            if n > 50:
                got_it = True
        else:
            print('  Not a valid zip.', flush=True)
    else:
        size = zip_path.stat().st_size if zip_path.exists() else 0
        print(f'  Download too small ({size} bytes).', flush=True)

    # Attempt 2: Internet Archive mirror
    if not got_it:
        print('\n── Attempt 2: Internet Archive mirror', flush=True)
        zip_path2 = staging / 'montgomery_ia.zip'
        run_live(f'curl -L --progress-bar -o {zip_path2} '
                 f'"https://archive.org/download/NLMMontgomeryCXRSet/NLM-MontgomeryCXRSet.zip"')
        if zip_path2.exists() and zip_path2.stat().st_size > 500_000:
            with open(zip_path2, 'rb') as f:
                magic = f.read(2)
            if magic == b'PK':
                print(f'  Valid zip ({zip_path2.stat().st_size/1e6:.1f} MB). Extracting:', flush=True)
                run_live(f'unzip -o {zip_path2} -d {staging}')
                n = count_images(staging)
                if n > 50:
                    got_it = True
            else:
                print('  Not a valid zip.', flush=True)
        else:
            size = zip_path2.stat().st_size if zip_path2.exists() else 0
            print(f'  Download too small ({size} bytes).', flush=True)

    if got_it:
        n_staged = count_images(staging)
        print(f'\n  Got {n_staged} Montgomery images. Copying to Drive...', flush=True)
        copy_to_drive(staging, dest)
        shutil.rmtree(staging, ignore_errors=True)
        print(f'Done: {count_images(dest)} total images in shenzhen_montgomery/.')
    else:
        shutil.rmtree(staging, ignore_errors=True)
        print('\nBoth downloads failed. Manual option:')
        print('  1. Go to kaggle.com and search "montgomery chest xray"')
        print('  2. Download the dataset')
        print('  3. Upload images to Drive: conformal-tb-triage/data/raw/shenzhen_montgomery/')
        print(f'  Need ~138 images to reach ~800 total (currently {total}).')

In [ ]:
# ── Cell 4: Download TBX11K ───────────────────────────────────────────
# Source: Kaggle (vbookshelf/tbx11k-simplified). ~3.8 GB.
# Downloads to local disk first (fast), then copies to Drive.

dest = RAW / 'tbx11k'
n = count_images(dest)

if n >= 5000:
    print(f'Already downloaded ({n} images). Skipping.')
else:
    print('Downloading TBX11K from Kaggle (~3.8 GB)...', flush=True)
    print('Kaggle progress bar should appear below:\n', flush=True)
    staging = TMP / 'tbx11k'
    shutil.rmtree(staging, ignore_errors=True)

    # kaggle_download uses os.system — progress prints live
    if kaggle_download('vbookshelf/tbx11k-simplified', staging):
        n_staged = count_images(staging)
        print(f'\nDownloaded {n_staged} images to local disk.', flush=True)
        print('Copying to Google Drive (this is the slow part)...', flush=True)
        copy_to_drive(staging, dest)
        shutil.rmtree(staging, ignore_errors=True)
        print(f'Done: {count_images(dest)} images on Drive.')
    else:
        print('\nDownload failed. Check your Kaggle token and try again.')
        print('Manual download: https://www.kaggle.com/datasets/vbookshelf/tbx11k-simplified')

In [ ]:
# ── Cell 5: Download CheXpert-small + create 5K subset ────────────────
# Source: Kaggle. ~11 GB download, but we keep only 5,000 random images.
# The full download goes to local Colab disk, subset goes to Drive.

dest = RAW / 'chexpert_subset'
n = count_images(dest)

if n >= 4900:
    print(f'Already have {n} images. Skipping.')
else:
    print('Downloading CheXpert-small from Kaggle (~11 GB)...', flush=True)
    print('This is the largest download — expect 10-20 minutes.', flush=True)
    print('Kaggle progress bar should appear below:\n', flush=True)
    staging = TMP / 'chexpert_full'
    shutil.rmtree(staging, ignore_errors=True)

    if kaggle_download('ashery/chexpert', staging):
        # Find all jpg images
        print('\nScanning downloaded files...', flush=True)
        all_images = sorted(staging.rglob('*.jpg'))
        print(f'Found {len(all_images)} images total.', flush=True)

        # Try to pick frontal views if identifiable
        frontal = [p for p in all_images if 'frontal' in str(p).lower()]
        pool = frontal if len(frontal) > 5000 else all_images
        if frontal:
            print(f'  ({len(frontal)} are frontal views — sampling from those)', flush=True)

        # Random 5,000 subset (seed 42 for reproducibility)
        random.seed(42)
        sample = random.sample(pool, min(5000, len(pool)))

        print(f'Copying {len(sample)}-image subset to Drive...', flush=True)
        dest.mkdir(parents=True, exist_ok=True)
        for i, src in enumerate(sample):
            target = dest / f'chexpert_{i:05d}.jpg'
            if not target.exists():
                shutil.copy2(src, target)
            if (i + 1) % 500 == 0 or (i + 1) == len(sample):
                print(f'  {i+1}/{len(sample)} images copied...', flush=True)

        # Free local disk
        print('Cleaning up local staging (~11 GB)...', flush=True)
        shutil.rmtree(staging, ignore_errors=True)
        print(f'Done: {count_images(dest)} images on Drive.')
    else:
        print('\nDownload failed. Try manually from:')
        print('  https://www.kaggle.com/datasets/ashery/chexpert')

In [ ]:
# ── Cell 6: Download Mendeley Pakistan TB ─────────────────────────────
# Automated download is unreliable. This cell tries, then falls back
# to manual upload.

dest = RAW / 'mendeley_pakistan'
n = count_images(dest)

if n >= 1000:
    print(f'Already have {n} images. Skipping.')
else:
    print('Attempting Mendeley Pakistan download...', flush=True)
    zip_path = TMP / 'mendeley_pakistan.zip'

    print('\n── Downloading with curl:', flush=True)
    run_live(
        f'curl -L --progress-bar -o {zip_path} '
        f'"https://data.mendeley.com/public-files/datasets/8j2g3csprk/files/d8a1c555-b178-4d81-98e0-76a41a54dab0/file_downloaded"'
    )

    # Check if we got a real zip
    got_zip = False
    if zip_path.exists():
        size = zip_path.stat().st_size
        print(f'\n  Downloaded file size: {size/1e6:.1f} MB', flush=True)
        if size > 1_000_000:
            with open(zip_path, 'rb') as f:
                if f.read(2) == b'PK':
                    got_zip = True
                    print('  Valid zip file.', flush=True)
                else:
                    print('  NOT a zip (probably HTML error page).', flush=True)
        else:
            print('  Too small to be the dataset.', flush=True)
    else:
        print('\n  No file downloaded.', flush=True)

    if got_zip:
        staging = TMP / 'mendeley_pak'
        print('\n── Extracting:', flush=True)
        run_live(f'unzip -o {zip_path} -d {staging}')
        print('\n── Copying to Drive:', flush=True)
        copy_to_drive(staging, dest)
        shutil.rmtree(staging, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
        print(f'\nDone: {count_images(dest)} images on Drive.')
    else:
        zip_path.unlink(missing_ok=True)
        print('\nAutomated download failed (common with Mendeley).')
        print()
        print('Manual steps:')
        print('  1. Go to: https://data.mendeley.com/datasets/8j2g3csprk/2')
        print('  2. Click the download button')
        print('  3. Upload the zip to Google Drive at:')
        print(f'     My Drive/conformal-tb-triage/data/raw/mendeley_pakistan/')
        print('  4. Or uncomment + run Cell 7 below to upload directly.')

In [ ]:
# ── Cell 9b: Build split manifest (FINAL — with correct labels) ──────
# Creates splits.parquet with proper TB labels from metadata files.
# Run ONCE. The SHA-256 hash goes into the pre-registration.

!pip install -q pandas pyarrow scikit-learn

import pandas as pd
import numpy as np
import hashlib
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
frames = []

# ── 1. SHENZHEN (from shenzhen_metadata.csv) ─────────────────────────

print('Processing Shenzhen...', flush=True)
shen_csv = RAW / 'shenzhen_montgomery' / 'shenzhen_metadata.csv'
shen_meta = pd.read_csv(shen_csv)

# Match metadata to actual image files
shen_img_dir = RAW / 'shenzhen_montgomery' / 'images' / 'images'
shen_rows = []
for _, row in shen_meta.iterrows():
    fname = row['study_id']
    img_path = shen_img_dir / fname
    if img_path.exists():
        is_normal = str(row['findings']).strip().lower() == 'normal'
        shen_rows.append({
            'patient_id': Path(fname).stem,
            'filename': fname,
            'file_path': str(img_path),
            'dataset': 'shenzhen',
            'label': 'normal' if is_normal else 'tb',
            'tb_binary': 'tb_negative' if is_normal else 'tb_positive',
            'sex': row.get('sex', 'Unknown'),
            'age': row.get('age', None),
        })

shen_df = pd.DataFrame(shen_rows)
tb_counts = shen_df['tb_binary'].value_counts().to_dict()
print(f'  {len(shen_df)} images: {tb_counts}', flush=True)
frames.append(shen_df)

# ── 2. MONTGOMERY (label from filename suffix: _0=normal, _1=TB) ─────

print('Processing Montgomery...', flush=True)
mont_img_dir = RAW / 'shenzhen_montgomery' / 'MontgomerySet' / 'CXR_png'
mont_rows = []
if mont_img_dir.exists():
    for img in sorted(mont_img_dir.glob('*.png')):
        fname = img.name
        # MCUCXR_0001_0.png → last digit before .png is the label
        stem = img.stem  # e.g. MCUCXR_0001_0
        label_digit = stem.split('_')[-1]  # '0' or '1'
        is_tb = label_digit == '1'
        mont_rows.append({
            'patient_id': stem,
            'filename': fname,
            'file_path': str(img),
            'dataset': 'montgomery',
            'label': 'tb' if is_tb else 'normal',
            'tb_binary': 'tb_positive' if is_tb else 'tb_negative',
            'sex': 'Unknown',
            'age': None,
        })

mont_df = pd.DataFrame(mont_rows)
tb_counts = mont_df['tb_binary'].value_counts().to_dict()
print(f'  {len(mont_df)} images: {tb_counts}', flush=True)
frames.append(mont_df)

# ── 3. TBX11K (from data.csv) ────────────────────────────────────────

print('Processing TBX11K...', flush=True)
tbx_csv = RAW / 'tbx11k' / 'tbx11k-simplified' / 'data.csv'
tbx_meta = pd.read_csv(tbx_csv)

# Images are in two folders: images/ and test/
tbx_base = RAW / 'tbx11k' / 'tbx11k-simplified'
tbx_rows = []
matched = 0
unmatched_files = []

for _, row in tbx_meta.iterrows():
    fname = row['fname']
    # Check both folders
    img_path = tbx_base / 'images' / fname
    if not img_path.exists():
        img_path = tbx_base / 'test' / fname
    if not img_path.exists():
        continue
    matched += 1

    target = str(row['target']).strip().lower()
    image_type = str(row.get('image_type', '')).strip().lower()
    tb_type = str(row.get('tb_type', '')).strip().lower()

    # Multi-class label (for secondary analyses)
    if image_type == 'healthy':
        label = 'healthy'
    elif image_type == 'sick_but_no_tb':
        label = 'sick_non_tb'
    elif tb_type == 'active_tb':
        label = 'active_tb'
    elif tb_type == 'latent_tb':
        label = 'latent_tb'
    elif target == 'tb':
        label = 'tb'
    else:
        label = 'no_tb'

    tb_binary = 'tb_positive' if target == 'tb' else 'tb_negative'

    tbx_rows.append({
        'patient_id': Path(fname).stem,
        'filename': fname,
        'file_path': str(img_path),
        'dataset': 'tbx11k',
        'label': label,
        'tb_binary': tb_binary,
        'sex': 'Unknown',
        'age': None,
    })

tbx_df = pd.DataFrame(tbx_rows)
print(f'  {len(tbx_df)} images matched to CSV ({len(tbx_meta)} rows in CSV)', flush=True)
print(f'  Binary: {tbx_df["tb_binary"].value_counts().to_dict()}', flush=True)
print(f'  Multi-class: {tbx_df["label"].value_counts().to_dict()}', flush=True)

# Check for images in test/ folder not in CSV
test_dir = tbx_base / 'test'
if test_dir.exists():
    test_files = set(p.name for p in test_dir.glob('*.png'))
    csv_files = set(tbx_meta['fname'].values)
    unlabelled = test_files - csv_files
    if unlabelled:
        print(f'  {len(unlabelled)} test images not in CSV (will be labelled from filename prefix)', flush=True)
        for fname in sorted(unlabelled):
            img_path = test_dir / fname
            # TBX11K naming: h=healthy, s=sick_non_tb, t=tb
            prefix = fname[0].lower()
            if prefix == 'h':
                label, tb_binary = 'healthy', 'tb_negative'
            elif prefix == 's':
                label, tb_binary = 'sick_non_tb', 'tb_negative'
            elif prefix == 't':
                label, tb_binary = 'tb', 'tb_positive'
            else:
                label, tb_binary = 'unknown', 'unknown'
            tbx_rows.append({
                'patient_id': Path(fname).stem,
                'filename': fname,
                'file_path': str(img_path),
                'dataset': 'tbx11k',
                'label': label,
                'tb_binary': tb_binary,
                'sex': 'Unknown',
                'age': None,
            })
        tbx_df = pd.DataFrame(tbx_rows)
        print(f'  After adding unlabelled: {len(tbx_df)} total TBX11K images', flush=True)
        print(f'  Binary: {tbx_df["tb_binary"].value_counts().to_dict()}', flush=True)

frames.append(tbx_df)

# ── 4. CHEXPERT SUBSET (all non-TB) ──────────────────────────────────

print('Processing CheXpert subset...', flush=True)
chx_dir = RAW / 'chexpert_subset'
chx_rows = []
if chx_dir.exists():
    for img in sorted(chx_dir.rglob('*.jpg')):
        chx_rows.append({
            'patient_id': img.stem,
            'filename': img.name,
            'file_path': str(img),
            'dataset': 'chexpert',
            'label': 'non_tb_distractor',
            'tb_binary': 'tb_negative',
            'sex': 'Unknown',
            'age': None,
        })
chx_df = pd.DataFrame(chx_rows)
print(f'  {len(chx_df)} images', flush=True)
frames.append(chx_df)

# ── 5. MENDELEY PAKISTAN (from folder structure) ──────────────────────

print('Processing Mendeley Pakistan...', flush=True)
pak_dir = RAW / 'mendeley_pakistan'
pak_rows = []
if pak_dir.exists():
    for img in sorted(pak_dir.rglob('*')):
        if img.suffix.lower() in ('.png', '.jpg', '.jpeg') and img.is_file():
            path_lower = str(img).lower()
            if '/normal' in path_lower:
                label, tb_binary = 'normal', 'tb_negative'
            elif '/tb' in path_lower or '/tuberculosis' in path_lower:
                label, tb_binary = 'tb', 'tb_positive'
            else:
                label, tb_binary = 'unknown', 'unknown'
            pak_rows.append({
                'patient_id': img.stem,
                'filename': img.name,
                'file_path': str(img),
                'dataset': 'pakistan',
                'label': label,
                'tb_binary': tb_binary,
                'sex': 'Unknown',
                'age': None,
            })
pak_df = pd.DataFrame(pak_rows)
print(f'  {len(pak_df)} images: {pak_df["tb_binary"].value_counts().to_dict()}', flush=True)
frames.append(pak_df)

# ── 6. COMBINE AND ASSIGN SPLITS ─────────────────────────────────────

all_images = pd.concat(frames, ignore_index=True)
print(f'\nTotal images: {len(all_images)}', flush=True)

# Deduplicate by patient_id
dupes = all_images['patient_id'].duplicated().sum()
if dupes > 0:
    print(f'Removing {dupes} duplicate patient IDs...', flush=True)
    all_images = all_images.drop_duplicates(subset='patient_id', keep='first').reset_index(drop=True)
    print(f'After dedup: {len(all_images)}', flush=True)

# Assign splits
all_images['split'] = 'unassigned'

# Calibration: Shenzhen + Montgomery
cal_mask = all_images['dataset'].isin(['shenzhen', 'montgomery'])
all_images.loc[cal_mask, 'split'] = 'calibration'

# Distractor: CheXpert
all_images.loc[all_images['dataset'] == 'chexpert', 'split'] = 'distractor'

# External: Pakistan
all_images.loc[all_images['dataset'] == 'pakistan', 'split'] = 'ext_pakistan'

# TBX11K: stratified 30% dev / 70% test
tbx_mask = all_images['dataset'] == 'tbx11k'
tbx_idx = all_images[tbx_mask].index
tbx_labels = all_images.loc[tbx_idx, 'tb_binary']

print(f'\nSplitting TBX11K ({len(tbx_idx)} images) into 30% dev / 70% test, stratified by TB status...', flush=True)
dev_idx, test_idx = train_test_split(
    tbx_idx, test_size=0.7, random_state=SEED,
    stratify=tbx_labels
)
all_images.loc[dev_idx, 'split'] = 'dev'
all_images.loc[test_idx, 'split'] = 'test'

# ── 7. SUMMARY ───────────────────────────────────────────────────────

print('\n' + '=' * 70, flush=True)
print('SPLIT SUMMARY', flush=True)
print('=' * 70, flush=True)

summary = all_images.groupby('split').agg(
    n=('patient_id', 'count'),
    tb_pos=('tb_binary', lambda x: (x == 'tb_positive').sum()),
    tb_neg=('tb_binary', lambda x: (x == 'tb_negative').sum()),
    unknown=('tb_binary', lambda x: (x == 'unknown').sum()),
    datasets=('dataset', lambda x: ', '.join(sorted(x.unique()))),
).reset_index()
print(summary.to_string(index=False), flush=True)

print('\nCalibration set label detail:', flush=True)
cal = all_images[all_images['split'] == 'calibration']
print(cal.groupby('dataset')['tb_binary'].value_counts().to_string(), flush=True)

print('\nTBX11K multi-class in test set:', flush=True)
test = all_images[all_images['split'] == 'test']
print(test[test['dataset'] == 'tbx11k']['label'].value_counts().to_string(), flush=True)

# ── 8. SAVE ──────────────────────────────────────────────────────────

output_dir = DRIVE_ROOT / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'splits.parquet'

cols = ['patient_id', 'filename', 'file_path', 'dataset', 'label', 'tb_binary', 'split', 'sex', 'age']
all_images[cols].to_parquet(output_path, index=False)
print(f'\nSaved: {output_path}', flush=True)
print(f'Rows:  {len(all_images)}', flush=True)

# ── 9. SHA-256 HASH ──────────────────────────────────────────────────

h = hashlib.sha256()
with open(output_path, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
sha = h.hexdigest()

print(f'\n{"=" * 60}', flush=True)
print(f'SPLIT MANIFEST SHA-256:', flush=True)
print(f'  {sha}', flush=True)
print(f'{"=" * 60}', flush=True)
print(f'\nPaste this hash into osf/registration.md at:')
print(f'  **Split manifest hash:** {sha}')
print(f'\nThen upload registration.md to OSF and freeze.')

In [ ]:
# ── Cell 9c: Inspect dataset folder structures ───────────────────────
# We need to find label files (CSV, JSON, TXT) and understand the
# directory layout to assign TB labels correctly.

print('=' * 60)
print('SHENZHEN / MONTGOMERY')
print('=' * 60)

sm_path = RAW / 'shenzhen_montgomery'

# Show directory tree (top 3 levels)
print('\nDirectory structure (top 3 levels):')
for p in sorted(sm_path.rglob('*')):
    depth = len(p.relative_to(sm_path).parts)
    if depth <= 3 and (p.is_dir() or p.suffix.lower() in ('.csv', '.json', '.txt', '.xlsx', '.xml')):
        prefix = '  ' * depth
        if p.is_dir():
            n_files = sum(1 for _ in p.iterdir()) if p.is_dir() else 0
            print(f'{prefix}{p.name}/ ({n_files} items)')
        else:
            print(f'{prefix}{p.name} ({p.stat().st_size/1e3:.1f} KB)')

# Show any non-image files
print('\nNon-image files (potential label sources):')
for p in sorted(sm_path.rglob('*')):
    if p.is_file() and p.suffix.lower() not in ('.png', '.jpg', '.jpeg', '.dcm', '.zip'):
        print(f'  {p.relative_to(sm_path)} ({p.stat().st_size/1e3:.1f} KB)')

# Show a few sample image filenames
print('\nSample image filenames (first 10):')
imgs = sorted(sm_path.rglob('*.png'))[:10] + sorted(sm_path.rglob('*.jpg'))[:10]
for p in imgs[:10]:
    print(f'  {p.relative_to(sm_path)}')

print('\n' + '=' * 60)
print('TBX11K')
print('=' * 60)

tbx_path = RAW / 'tbx11k'

print('\nDirectory structure (top 3 levels):')
for p in sorted(tbx_path.rglob('*')):
    depth = len(p.relative_to(tbx_path).parts)
    if depth <= 3 and (p.is_dir() or p.suffix.lower() in ('.csv', '.json', '.txt', '.xlsx', '.xml')):
        prefix = '  ' * depth
        if p.is_dir():
            n_files = sum(1 for _ in p.iterdir()) if p.is_dir() else 0
            print(f'{prefix}{p.name}/ ({n_files} items)')
        else:
            print(f'{prefix}{p.name} ({p.stat().st_size/1e3:.1f} KB)')

print('\nNon-image files (potential label sources):')
for p in sorted(tbx_path.rglob('*')):
    if p.is_file() and p.suffix.lower() not in ('.png', '.jpg', '.jpeg', '.dcm', '.zip'):
        print(f'  {p.relative_to(tbx_path)} ({p.stat().st_size/1e3:.1f} KB)')

print('\nSample image filenames (first 10):')
imgs = sorted(tbx_path.rglob('*.png'))[:10] + sorted(tbx_path.rglob('*.jpg'))[:10]
for p in imgs[:10]:
    print(f'  {p.relative_to(tbx_path)}')

In [ ]:
# ── Cell 9d: Peek at label files ──────────────────────────────────────

import pandas as pd

print('=' * 60)
print('SHENZHEN METADATA')
print('=' * 60)
shen_csv = RAW / 'shenzhen_montgomery' / 'shenzhen_metadata.csv'
if shen_csv.exists():
    df = pd.read_csv(shen_csv)
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'\nFirst 5 rows:')
    print(df.head().to_string())
    print(f'\nLabel-like column value counts:')
    for col in df.columns:
        if df[col].nunique() < 20:
            print(f'  {col}: {df[col].value_counts().to_dict()}')
else:
    print(f'NOT FOUND: {shen_csv}')

print('\n' + '=' * 60)
print('TBX11K DATA.CSV')
print('=' * 60)
tbx_csv = RAW / 'tbx11k' / 'tbx11k-simplified' / 'data.csv'
if tbx_csv.exists():
    df = pd.read_csv(tbx_csv)
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'\nFirst 5 rows:')
    print(df.head().to_string())
    print(f'\nLabel-like column value counts:')
    for col in df.columns:
        if df[col].nunique() < 20:
            print(f'  {col}: {df[col].value_counts().to_dict()}')
else:
    print(f'NOT FOUND: {tbx_csv}')

In [ ]:
# ── Cell 8: Verification ──────────────────────────────────────────────
# Shows what's on Drive and what's still missing.

checks = [
    ('Shenzhen + Montgomery', RAW/'shenzhen_montgomery', 600, 'auto (Cell 3)'),
    ('TBX11K',               RAW/'tbx11k',              5000, 'auto (Cell 4)'),
    ('CheXpert (5K subset)', RAW/'chexpert_subset',      4900, 'auto (Cell 5)'),
    ('Mendeley Pakistan',    RAW/'mendeley_pakistan',     1000, 'auto/manual (Cell 6-7)'),
    ('JSRT',                 RAW/'jsrt',                 200,  'manual: db.jsrt.or.jp'),
    ('PadChest (subset)',    RAW/'padchest_subset',       4900, 'manual: bimcv.cipf.es'),
    ('NIH CXR14 (subset)',   RAW/'nih_cxr14',            4900, 'manual: Kaggle Notebook'),
    ('TB Portals',           RAW/'tb_portals',            100,  'pending DUA from NIAID'),
]

print(f'{"Dataset":<25} {"Found":<8} {"Need":<8} {"Status":<10} Source')
print('─' * 75)
for name, path, need, source in checks:
    n = count_images(path)
    if n >= need:
        status = 'OK'
    elif n > 0:
        status = 'PARTIAL'
    else:
        status = 'MISSING'
    print(f'{name:<25} {n:<8} {need:<8} {status:<10} {source}')

# Drive space
import shutil as _sh
total, used, free = _sh.disk_usage('/content/drive')
print(f'\nDrive: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free')
print(f'Timestamp: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')

In [ ]:
# ── Cell 8: Verification ──────────────────────────────────────────────
# Shows what's on Drive and what's still missing.

checks = [
    ('Shenzhen + Montgomery', RAW/'shenzhen_montgomery', 700, 'auto (Cell 3)'),
    ('TBX11K',               RAW/'tbx11k',              5000, 'auto (Cell 4)'),
    ('CheXpert (5K subset)', RAW/'chexpert_subset',      4900, 'auto (Cell 5)'),
    ('Mendeley Pakistan',    RAW/'mendeley_pakistan',     1000, 'auto/manual (Cell 6-7)'),
    ('JSRT',                 RAW/'jsrt',                 200,  'manual: db.jsrt.or.jp'),
    ('PadChest (subset)',    RAW/'padchest_subset',       4900, 'manual: bimcv.cipf.es'),
    ('NIH CXR14 (subset)',   RAW/'nih_cxr14',            4900, 'manual: Kaggle Notebook'),
    ('TB Portals',           RAW/'tb_portals',            100,  'pending DUA from NIAID'),
]

print(f'{"Dataset":<25} {"Found":<8} {"Need":<8} {"Status":<10} Source')
print('─' * 75)
for name, path, need, source in checks:
    n = count_images(path)
    if n >= need:
        status = 'OK'
    elif n > 0:
        status = 'PARTIAL'
    else:
        status = 'MISSING'
    print(f'{name:<25} {n:<8} {need:<8} {status:<10} {source}')

# Drive space
import shutil as _sh
total, used, free = _sh.disk_usage('/content/drive')
print(f'\nDrive: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free')
print(f'Timestamp: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')

In [ ]:
# ── Cell 9: SHA-256 hashes for data snapshot log ──────────────────────
# Run after all downloads are complete. Copy output into
# osf/data_snapshot_log.md.

def hash_dir(d):
    d = Path(d)
    if not d.exists(): return 'NOT_FOUND'
    imgs = sorted(p for p in d.rglob('*')
                  if p.suffix.lower() in ('.png','.jpg','.jpeg','.dcm'))
    if not imgs: return 'NO_IMAGES'
    h = hashlib.sha256()
    for f in imgs:
        h.update(f.name.encode())
        h.update(str(f.stat().st_size).encode())
    return h.hexdigest()

print('Dataset content hashes (filename + size):')
print('=' * 75)
for name, path, _, _ in checks:
    print(f'{name:<25} {hash_dir(path)}')
print(f'\nTimestamp: {datetime.now(timezone.utc).isoformat()}')

---
## Still needed (manual)

| Dataset | How | Put files in |
|---------|-----|-------------|
| **JSRT** | Register at http://db.jsrt.or.jp/eng.php — link emailed | `conformal-tb-triage/data/raw/jsrt/` |
| **PadChest** | Register at bimcv.cipf.es — download 1-2 zips, extract 5K PA images | `conformal-tb-triage/data/raw/padchest_subset/` |
| **NIH CXR14** | 42 GB. Use a Kaggle Notebook with `nih-chest-xrays/data` attached, sample 5K | `conformal-tb-triage/data/raw/nih_cxr14/` |
| **TB Portals** | Pending DUA from accessclinicaldata@NIAID.nih.gov | `conformal-tb-triage/data/raw/tb_portals/` |

Upload to these Drive folders from your browser any time. Re-run Cell 8 to check progress.

**Next notebook:** `02_extract_embeddings.ipynb` (T4 GPU runtime).